In [1]:
# ============================================================
# M1-B FULL CORRECTED DATASET VERIFICATION
# CELL 1: SETUP AND CONFIGURATION
# ============================================================

import os
import re
import json
import hashlib
import unicodedata

import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

# ------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------

INPUT_ROOT = "/kaggle/input"

EXPECTED_ROWS = 106870
EXPECTED_RECORDINGS = 46
EXPECTED_PEOPLE = 45
EXPECTED_VALID_BEATS = 105047

AAMI_CLASSES = ["N", "S", "V", "F", "Q"]

EXPECTED_CLASS_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

# One person has two recordings: 201 and 202.
def get_person_id(record_id):

    record_id = int(record_id)

    if record_id in [201, 202]:
        return "person_201_202"

    return f"person_{record_id}"


# ------------------------------------------------------------
# Original annotation symbol to AAMI class mapping
# ------------------------------------------------------------

AAMI_MAP = {
    "N": "N",
    "L": "N",
    "R": "N",
    "e": "N",
    "j": "N",

    "A": "S",
    "a": "S",
    "J": "S",
    "S": "S",

    "V": "V",
    "E": "V",

    "F": "F",

    "/": "Q",
    "f": "Q",
    "Q": "Q"
}


# ------------------------------------------------------------
# Waveform columns
# ------------------------------------------------------------

MLII_COLUMNS = [
    f"MLII_{i}" for i in range(201)
]

LEAD2_COLUMNS = [
    f"Lead2_{i}" for i in range(201)
]

WAVEFORM_COLUMNS = MLII_COLUMNS + LEAD2_COLUMNS

assert len(MLII_COLUMNS) == 201
assert len(LEAD2_COLUMNS) == 201
assert len(WAVEFORM_COLUMNS) == 402


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def clean_column_name(name):

    name = unicodedata.normalize("NFKC", str(name))

    return (
        name.strip()
        .strip('"')
        .strip("'")
        .strip()
    )


def normalise_name(name):

    name = clean_column_name(name).casefold()

    return re.sub(r"[^a-z0-9#]", "", name)


def get_record_id_from_filename(filename):

    stem = Path(filename).stem

    match = re.match(r"^(\d{3})(?:\D|$)", stem)

    if match:
        return int(match.group(1))

    return None


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def waveform_digest(values):

    values = np.asarray(values, dtype=np.float64)

    values = np.ascontiguousarray(values)

    return hashlib.blake2b(
        values.tobytes(),
        digest_size=16
    ).hexdigest()


print("=" * 75)
print("M1-B: FULL CORRECTED DATASET VERIFICATION")
print("=" * 75)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\nExpected recordings:", EXPECTED_RECORDINGS)
print("Expected people:", EXPECTED_PEOPLE)
print("Expected valid beats:", EXPECTED_VALID_BEATS)
print("Expected waveform values per beat:", len(WAVEFORM_COLUMNS))

print("\nSetup complete.")
print("No input datasets have been modified.") 

M1-B: FULL CORRECTED DATASET VERIFICATION
NumPy: 2.1.3
Pandas: 2.3.3

Expected recordings: 46
Expected people: 45
Expected valid beats: 105047
Expected waveform values per beat: 402

Setup complete.
No input datasets have been modified.


In [2]:
# ============================================================
# CELL 2: LOCATE AND VALIDATE ORIGINAL + CORRECTED DATASETS
# ============================================================

# ------------------------------------------------------------
# 1. Discover the original and corrected CSV files
# ------------------------------------------------------------

original_candidates = []
corrected_candidates = []

for root, dirs, files in os.walk(INPUT_ROOT):

    for filename in files:

        if not filename.lower().endswith(".csv"):
            continue

        full_path = os.path.join(root, filename)

        lower_name = filename.lower()

        try:

            header = pd.read_csv(full_path, nrows=0)

            headers = {
                clean_column_name(c)
                for c in header.columns
            }

            required_columns = {
                "Record_ID",
                "Label",
                "MLII_0",
                "MLII_200",
                "Lead2_0",
                "Lead2_200"
            }

            if not required_columns.issubset(headers):
                continue

            if lower_name == "mitbih_mlii_duallead_dataset.csv":

                original_candidates.append(full_path)

            elif "corrected" in lower_name:

                corrected_candidates.append(full_path)

        except Exception as e:

            print("Could not inspect:", full_path, "|", e)


print("ORIGINAL DATASET CANDIDATES:")
for p in original_candidates:
    print(p)

print("\nCORRECTED DATASET CANDIDATES:")
for p in corrected_candidates:
    print(p)


if len(original_candidates) != 1:

    raise RuntimeError(
        f"Expected exactly one original processed dataset. "
        f"Found {len(original_candidates)}. "
        "Check that the original input dataset is attached."
    )


if len(corrected_candidates) != 1:

    raise RuntimeError(
        f"Expected exactly one corrected processed CSV. "
        f"Found {len(corrected_candidates)}. "
        "Check that the corrected Kaggle dataset is attached "
        "and that its filename contains 'corrected'."
    )


original_dataset_path = original_candidates[0]
corrected_dataset_path = corrected_candidates[0]


# ------------------------------------------------------------
# 2. Load both datasets
# ------------------------------------------------------------

print("\nLoading original dataset...")

original_df = pd.read_csv(
    original_dataset_path,
    low_memory=False
)

print("Loading corrected dataset...")

corrected_df = pd.read_csv(
    corrected_dataset_path,
    low_memory=False
)


# Clean headers
original_df.columns = [
    clean_column_name(c)
    for c in original_df.columns
]

corrected_df.columns = [
    clean_column_name(c)
    for c in corrected_df.columns
]


# Convert Record_ID to integer
original_df["Record_ID"] = pd.to_numeric(
    original_df["Record_ID"],
    errors="raise"
).astype(int)

corrected_df["Record_ID"] = pd.to_numeric(
    corrected_df["Record_ID"],
    errors="raise"
).astype(int)


print("\nDATASET DIMENSIONS")
print("Original:", original_df.shape)
print("Corrected:", corrected_df.shape)


# ------------------------------------------------------------
# 3. Basic structure checks
# ------------------------------------------------------------

if original_df.shape != (EXPECTED_ROWS, 404):
    raise ValueError(
        f"Unexpected original shape: {original_df.shape}"
    )

if corrected_df.shape != (EXPECTED_ROWS, 404):
    raise ValueError(
        f"Unexpected corrected shape: {corrected_df.shape}"
    )

if original_df.columns.tolist() != corrected_df.columns.tolist():
    raise ValueError(
        "Original and corrected column names/order do not match."
    )

missing_columns = [
    c for c in WAVEFORM_COLUMNS
    if c not in corrected_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing waveform columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 4. Verify Record IDs and labels have not changed
# ------------------------------------------------------------

record_ids_match = original_df["Record_ID"].equals(
    corrected_df["Record_ID"]
)

labels_match = original_df["Label"].equals(
    corrected_df["Label"]
)

column_order_match = (
    original_df.columns.tolist()
    == corrected_df.columns.tolist()
)

print("\nSTRUCTURE AND METADATA")
print("Record IDs unchanged:", record_ids_match)
print("Labels unchanged:", labels_match)
print("Column order unchanged:", column_order_match)


# ------------------------------------------------------------
# 5. Verify records other than 114 are unchanged
# ------------------------------------------------------------

other_records_mask = (
    original_df["Record_ID"] != 114
)

original_other = (
    original_df.loc[other_records_mask]
    .reset_index(drop=True)
)

corrected_other = (
    corrected_df.loc[other_records_mask]
    .reset_index(drop=True)
)

other_records_unchanged = original_other.equals(
    corrected_other
)

print(
    "Other 45 recordings unchanged:",
    other_records_unchanged
)


# ------------------------------------------------------------
# 6. Verify Record 114 was correctly swapped
# ------------------------------------------------------------

original_114 = original_df[
    original_df["Record_ID"] == 114
]

corrected_114 = corrected_df[
    corrected_df["Record_ID"] == 114
]

record114_row_count_match = (
    len(original_114) == len(corrected_114) == 1887
)

record114_mlii_correct = np.array_equal(
    corrected_114[MLII_COLUMNS].to_numpy(),
    original_114[LEAD2_COLUMNS].to_numpy()
)

record114_lead2_correct = np.array_equal(
    corrected_114[LEAD2_COLUMNS].to_numpy(),
    original_114[MLII_COLUMNS].to_numpy()
)

print("\nRECORD 114 CORRECTION")
print("Original rows:", len(original_114))
print("Corrected rows:", len(corrected_114))
print("MLII values correctly exchanged:", record114_mlii_correct)
print("Lead2 values correctly exchanged:", record114_lead2_correct)


# ------------------------------------------------------------
# 7. Correction provenance gate
# ------------------------------------------------------------

provenance_checks = {
    "Original_shape_correct": original_df.shape == (EXPECTED_ROWS, 404),
    "Corrected_shape_correct": corrected_df.shape == (EXPECTED_ROWS, 404),
    "Columns_and_order_match": column_order_match,
    "Record_IDs_unchanged": record_ids_match,
    "Labels_unchanged": labels_match,
    "Other_45_recordings_unchanged": other_records_unchanged,
    "Record114_row_count_correct": record114_row_count_match,
    "Record114_MLII_correct": record114_mlii_correct,
    "Record114_Lead2_correct": record114_lead2_correct
}

print("\n" + "=" * 75)
print("CORRECTION PROVENANCE CHECKS")
print("=" * 75)

for name, passed in provenance_checks.items():

    print(f"{name}: {'PASS' if passed else 'FAIL'}")


CORRECTION_PROVENANCE_PASSED = all(
    provenance_checks.values()
)

if not CORRECTION_PROVENANCE_PASSED:

    raise RuntimeError(
        "Corrected dataset provenance checks failed. "
        "Stop here; do not continue to reconstruction."
    )


# ------------------------------------------------------------
# 8. Identify processed recording IDs
# ------------------------------------------------------------

EXPECTED_RECORD_IDS = sorted(
    corrected_df["Record_ID"].unique().tolist()
)

print("\nProcessed recording IDs:", EXPECTED_RECORD_IDS)
print("Number of recordings:", len(EXPECTED_RECORD_IDS))

print("\nCELL 2 COMPLETE — CORRECTED DATASET PROVENANCE VERIFIED.")

ORIGINAL DATASET CANDIDATES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv

CORRECTED DATASET CANDIDATES:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/MITBIH_MLII_DualLead_Dataset_corrected.csv

Loading original dataset...
Loading corrected dataset...

DATASET DIMENSIONS
Original: (106870, 404)
Corrected: (106870, 404)

STRUCTURE AND METADATA
Record IDs unchanged: True
Labels unchanged: True
Column order unchanged: True
Other 45 recordings unchanged: True

RECORD 114 CORRECTION
Original rows: 1887
Corrected rows: 1887
MLII values correctly exchanged: True
Lead2 values correctly exchanged: True

CORRECTION PROVENANCE CHECKS
Original_shape_correct: PASS
Corrected_shape_correct: PASS
Columns_and_order_match: PASS
Record_IDs_unchanged: PASS
Labels_unchanged: PASS
Other_45_recordings_unchanged: PASS
Record114_row_count_correct: PASS
Record114_MLII_correct: PASS
Record1

In [3]:
# ============================================================
# CELL 3: FIND ALL 46 ORIGINAL SIGNAL AND ANNOTATION FILES
# ============================================================

signal_files = {}
annotation_files = {}

signal_candidates_by_id = {}
annotation_candidates_by_id = {}

for root, dirs, files in os.walk(INPUT_ROOT):

    folder_parts = {
        part.casefold()
        for part in Path(root).parts
    }

    is_signal_folder = (
        "signals" in folder_parts
    )

    is_annotation_folder = (
        "annotations" in folder_parts
    )

    for filename in files:

        record_id = get_record_id_from_filename(filename)

        if record_id is None:
            continue

        full_path = os.path.join(root, filename)

        # Original ECG signal files
        if (
            is_signal_folder
            and filename.lower().endswith(".csv")
        ):

            signal_candidates_by_id.setdefault(
                record_id, []
            ).append(full_path)


        # Original annotation text files
        if (
            is_annotation_folder
            and filename.lower().endswith(".txt")
        ):

            annotation_candidates_by_id.setdefault(
                record_id, []
            ).append(full_path)


# ------------------------------------------------------------
# 1. Match each processed recording with its source files
# ------------------------------------------------------------

missing_signal_records = []
missing_annotation_records = []
duplicate_signal_records = []
duplicate_annotation_records = []

for record_id in EXPECTED_RECORD_IDS:

    signal_matches = signal_candidates_by_id.get(
        record_id, []
    )

    annotation_matches = annotation_candidates_by_id.get(
        record_id, []
    )

    if len(signal_matches) == 0:
        missing_signal_records.append(record_id)

    elif len(signal_matches) > 1:
        duplicate_signal_records.append(
            (record_id, signal_matches)
        )

    else:
        signal_files[record_id] = signal_matches[0]


    if len(annotation_matches) == 0:
        missing_annotation_records.append(record_id)

    elif len(annotation_matches) > 1:
        duplicate_annotation_records.append(
            (record_id, annotation_matches)
        )

    else:
        annotation_files[record_id] = annotation_matches[0]


# ------------------------------------------------------------
# 2. Display source matching results
# ------------------------------------------------------------

print("=" * 75)
print("ORIGINAL SOURCE FILE MATCHING")
print("=" * 75)

print("Required recordings:", len(EXPECTED_RECORD_IDS))
print("Matched signal files:", len(signal_files))
print("Matched annotation files:", len(annotation_files))

print("\nMissing signal files:", missing_signal_records)
print("Missing annotation files:", missing_annotation_records)

print("\nDuplicate signal files:", duplicate_signal_records)
print("Duplicate annotation files:", duplicate_annotation_records)


if (
    missing_signal_records
    or missing_annotation_records
    or duplicate_signal_records
    or duplicate_annotation_records
):

    raise RuntimeError(
        "Source file matching failed. "
        "Do not proceed until the source files are resolved."
    )


# ------------------------------------------------------------
# 3. Inspect headers for all 46 signal files
# ------------------------------------------------------------

source_header_audit = []

for record_id in EXPECTED_RECORD_IDS:

    path = signal_files[record_id]

    header = pd.read_csv(
        path,
        nrows=0
    )

    cleaned_columns = [
        clean_column_name(c)
        for c in header.columns
    ]

    normalised_lookup = {
        normalise_name(c): c
        for c in cleaned_columns
    }

    sample_matches = [
        c for k, c in normalised_lookup.items()
        if k in ["sample#", "sample"]
    ]

    mlii_matches = [
        c for k, c in normalised_lookup.items()
        if k == "mlii"
    ]

    if len(sample_matches) != 1:
        raise ValueError(
            f"Record {record_id}: sample column not identified."
        )

    if len(mlii_matches) != 1:
        raise ValueError(
            f"Record {record_id}: MLII column not identified."
        )

    sample_column = sample_matches[0]
    mlii_column = mlii_matches[0]

    other_leads = [
        c for c in cleaned_columns
        if c not in [sample_column, mlii_column]
    ]

    if len(other_leads) != 1:
        raise ValueError(
            f"Record {record_id}: expected one second lead, "
            f"found {other_leads}."
        )

    source_header_audit.append({

        "Record_ID": record_id,
        "Signal_File": path,
        "Annotation_File": annotation_files[record_id],
        "Sample_Column": sample_column,
        "MLII_Column": mlii_column,
        "Second_Lead_Column": other_leads[0],
        "Header_Check": True
    })


source_header_audit_df = pd.DataFrame(
    source_header_audit
)

print("\nSOURCE HEADER AUDIT")
display(source_header_audit_df.head(10))

print("\nTotal eligible recordings with valid source headers:",
      len(source_header_audit_df))

print("\nCELL 3 COMPLETE — ALL 46 SOURCE PAIRS LOCATED.")

ORIGINAL SOURCE FILE MATCHING
Required recordings: 46
Matched signal files: 46
Matched annotation files: 46

Missing signal files: []
Missing annotation files: []

Duplicate signal files: []
Duplicate annotation files: []

SOURCE HEADER AUDIT


,Record_ID,Signal_File,Annotation_File,Sample_Column,MLII_Column,Second_Lead_Column,Header_Check
0,100,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V5,True
1,101,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
2,103,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V2,True
3,105,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
4,106,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
5,107,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
6,108,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
7,109,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
8,111,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True
9,112,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,/kaggle/input/datasets/aakashshah3799/ecg-arrh...,sample #,MLII,V1,True



Total eligible recordings with valid source headers: 46

CELL 3 COMPLETE — ALL 46 SOURCE PAIRS LOCATED.


In [4]:
# ============================================================
# CELL 4: RECONSTRUCT ALL VALID ECG BEATS
# ============================================================

reconstructed_parts = []

record_reconstruction_audit = []

all_parser_errors = []


# ------------------------------------------------------------
# Process one recording at a time
# ------------------------------------------------------------

for record_id in EXPECTED_RECORD_IDS:

    print(f"\nProcessing record {record_id}...")

    signal_path = signal_files[record_id]
    annotation_path = annotation_files[record_id]

    # --------------------------------------------------------
    # A. Load signal
    # --------------------------------------------------------

    signal_df = pd.read_csv(signal_path)

    signal_df.columns = [
        clean_column_name(c)
        for c in signal_df.columns
    ]

    normalised_lookup = {
        normalise_name(c): c
        for c in signal_df.columns
    }

    sample_col = normalised_lookup.get("sample#")

    if sample_col is None:
        sample_col = normalised_lookup.get("sample")

    mlii_col = normalised_lookup.get("mlii")

    if sample_col is None or mlii_col is None:
        raise ValueError(
            f"Record {record_id}: sample or MLII column missing."
        )

    other_leads = [
        c for c in signal_df.columns
        if c not in [sample_col, mlii_col]
    ]

    if len(other_leads) != 1:

        raise ValueError(
            f"Record {record_id}: expected one second ECG lead. "
            f"Found {other_leads}"
        )

    second_lead_col = other_leads[0]


    # Convert source signal values to numeric.
    samples = pd.to_numeric(
        signal_df[sample_col],
        errors="raise"
    ).to_numpy(dtype=np.int64)

    mlii_values = pd.to_numeric(
        signal_df[mlii_col],
        errors="raise"
    ).to_numpy(dtype=np.float64)

    second_values = pd.to_numeric(
        signal_df[second_lead_col],
        errors="raise"
    ).to_numpy(dtype=np.float64)


    if len(np.unique(samples)) != len(samples):

        raise ValueError(
            f"Record {record_id}: duplicate signal sample positions."
        )

    if not np.all(np.diff(samples) > 0):

        raise ValueError(
            f"Record {record_id}: signal sample indices are not "
            "strictly increasing."
        )

    signal_values = np.column_stack([
        mlii_values,
        second_values
    ])

    sample_to_position = {
        int(sample): position
        for position, sample in enumerate(samples)
    }


    # --------------------------------------------------------
    # B. Parse annotation file
    # --------------------------------------------------------

    annotation_rows = []
    parser_errors = []
    original_symbol_counts = Counter()
    excluded_symbol_counts = Counter()

    with open(
        annotation_path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            parts = re.split(
                r"\s+",
                line,
                maxsplit=6
            )

            if len(parts) < 3:
                continue

            try:

                sample_index = int(parts[1])

            except ValueError:

                # The annotation header is not a beat.
                if "sample" in line.casefold():
                    continue

                parser_errors.append({
                    "Record_ID": record_id,
                    "Line_Number": line_number,
                    "Line": line
                })

                continue


            original_symbol = parts[2]

            original_symbol_counts[
                original_symbol
            ] += 1


            # Keep only valid five-class AAMI beat symbols.
            if original_symbol in AAMI_MAP:

                annotation_rows.append({

                    "Record_ID": record_id,
                    "Person_ID": get_person_id(record_id),
                    "Sample_Index": sample_index,
                    "Original_Symbol": original_symbol,
                    "AAMI_Label": AAMI_MAP[original_symbol]

                })

            else:

                excluded_symbol_counts[
                    original_symbol
                ] += 1


    all_parser_errors.extend(parser_errors)

    annotation_df = pd.DataFrame(annotation_rows)


    # --------------------------------------------------------
    # C. Check annotation positions
    # --------------------------------------------------------

    if annotation_df.empty:

        raise ValueError(
            f"Record {record_id}: no valid AAMI beat annotations."
        )

    duplicate_annotation_keys = annotation_df.duplicated(
        ["Record_ID", "Sample_Index"]
    ).sum()

    if duplicate_annotation_keys > 0:

        raise ValueError(
            f"Record {record_id}: duplicate annotated beat positions."
        )


    # --------------------------------------------------------
    # D. Construct 201-sample windows around each beat
    # --------------------------------------------------------

    centres = annotation_df[
        "Sample_Index"
    ].to_numpy(dtype=np.int64)

    offsets = np.arange(
        -100,
        101,
        dtype=np.int64
    )

    required_samples = (
        centres[:, None] + offsets[None, :]
    )


    # Locate required samples in the source signal.
    positions = np.searchsorted(
        samples,
        required_samples
    )

    in_bounds = positions < len(samples)

    safe_positions = np.minimum(
        positions,
        len(samples) - 1
    )

    exact_sample_positions = (
        in_bounds
        & (
            samples[safe_positions]
            == required_samples
        )
    )

    complete_window = (
        exact_sample_positions.all(axis=1)
    )


    # Separate true recording boundaries from internal gaps.
    outside_signal = (
        (required_samples < samples.min())
        | (required_samples > samples.max())
    ).any(axis=1)

    boundary_skips = int(
        ((~complete_window) & outside_signal).sum()
    )

    missing_internal_windows = int(
        ((~complete_window) & (~outside_signal)).sum()
    )


    # Accept only complete 201-sample windows.
    accepted_annotation_df = (
        annotation_df.loc[complete_window]
        .reset_index(drop=True)
    )

    accepted_positions = positions[
        complete_window
    ]


    # --------------------------------------------------------
    # E. Extract both 201-sample leads
    # --------------------------------------------------------

    if len(accepted_positions) == 0:

        raise ValueError(
            f"Record {record_id}: no complete beat windows."
        )

    windows = signal_values[
        accepted_positions
    ]

    # Shape is:
    # number of beats x 201 samples x 2 leads

    extracted_mlii = windows[:, :, 0]

    extracted_second = windows[:, :, 1]


    # Convert the two leads into 402 columns.
    waveform_matrix = np.concatenate(
        [
            extracted_mlii,
            extracted_second
        ],
        axis=1
    )


    if waveform_matrix.shape[1] != 402:

        raise ValueError(
            f"Record {record_id}: incorrect waveform dimensions."
        )


    waveform_df = pd.DataFrame(
        waveform_matrix,
        columns=WAVEFORM_COLUMNS
    )


    # Create metadata table.
    metadata_df = accepted_annotation_df[
        [
            "Record_ID",
            "Person_ID",
            "Sample_Index",
            "Original_Symbol",
            "AAMI_Label"
        ]
    ].copy()


    reconstructed_record_df = pd.concat(
        [
            metadata_df.reset_index(drop=True),
            waveform_df
        ],
        axis=1
    )

    reconstructed_parts.append(
        reconstructed_record_df
    )


    # --------------------------------------------------------
    # F. Record-level audit
    # --------------------------------------------------------

    record_reconstruction_audit.append({

        "Record_ID": record_id,

        "Person_ID": get_person_id(record_id),

        "Original_MLII_Column": mlii_col,

        "Original_Second_Lead_Column": second_lead_col,

        "Raw_Annotation_Symbol_Count": int(
            sum(original_symbol_counts.values())
        ),

        "AAMI_Annotation_Candidates": len(annotation_df),

        "Accepted_Full_Windows": len(
            reconstructed_record_df
        ),

        "Boundary_Windows_Skipped": boundary_skips,

        "Missing_Internal_Windows": missing_internal_windows,

        "Excluded_Non_AAMI_Annotations": int(
            sum(excluded_symbol_counts.values())
        ),

        "Duplicate_Annotation_Keys": int(
            duplicate_annotation_keys
        ),

        "Parser_Errors": len(parser_errors)

    })


    print(
        f"Record {record_id}: "
        f"{len(reconstructed_record_df)} beats reconstructed; "
        f"{boundary_skips} boundary skips."
    )


# ------------------------------------------------------------
# Combine all recordings
# ------------------------------------------------------------

reconstructed_df = pd.concat(
    reconstructed_parts,
    ignore_index=True
)

record_reconstruction_audit_df = pd.DataFrame(
    record_reconstruction_audit
)

print("\n" + "=" * 75)
print("FULL RECONSTRUCTION COMPLETE")
print("=" * 75)

print("Reconstructed valid beats:", len(reconstructed_df))

print(
    "Recordings:",
    reconstructed_df["Record_ID"].nunique()
)

print(
    "People:",
    reconstructed_df["Person_ID"].nunique()
)

print(
    "Annotation parser errors:",
    len(all_parser_errors)
)

print("\nReconstructed class counts:")

display(
    reconstructed_df["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
    .to_frame("Count")
)

print("\nRECORD-LEVEL RECONSTRUCTION AUDIT:")

display(
    record_reconstruction_audit_df
)

print("\nCELL 4 COMPLETE.")


Processing record 100...
Record 100: 2271 beats reconstructed; 2 boundary skips.

Processing record 101...
Record 101: 1864 beats reconstructed; 1 boundary skips.

Processing record 103...
Record 103: 2084 beats reconstructed; 0 boundary skips.

Processing record 105...
Record 105: 2572 beats reconstructed; 0 boundary skips.

Processing record 106...
Record 106: 2027 beats reconstructed; 0 boundary skips.

Processing record 107...
Record 107: 2136 beats reconstructed; 1 boundary skips.

Processing record 108...
Record 108: 1762 beats reconstructed; 1 boundary skips.

Processing record 109...
Record 109: 2531 beats reconstructed; 1 boundary skips.

Processing record 111...
Record 111: 2124 beats reconstructed; 0 boundary skips.

Processing record 112...
Record 112: 2539 beats reconstructed; 0 boundary skips.

Processing record 113...
Record 113: 1794 beats reconstructed; 1 boundary skips.

Processing record 114...
Record 114: 1879 beats reconstructed; 0 boundary skips.

Processing reco

,Count
AAMI_Label,
N,90341
S,2781
V,7229
F,802
Q,3894



RECORD-LEVEL RECONSTRUCTION AUDIT:


,Record_ID,Person_ID,Original_MLII_Column,Original_Second_Lead_Column,Raw_Annotation_Symbol_Count,AAMI_Annotation_Candidates,Accepted_Full_Windows,Boundary_Windows_Skipped,Missing_Internal_Windows,Excluded_Non_AAMI_Annotations,Duplicate_Annotation_Keys,Parser_Errors
0,100,person_100,MLII,V5,2274,2273,2271,2,0,1,0,0
1,101,person_101,MLII,V1,1874,1865,1864,1,0,9,0,0
2,103,person_103,MLII,V2,2091,2084,2084,0,0,7,0,0
3,105,person_105,MLII,V1,2691,2572,2572,0,0,119,0,0
4,106,person_106,MLII,V1,2098,2027,2027,0,0,71,0,0
5,107,person_107,MLII,V1,2140,2137,2136,1,0,3,0,0
6,108,person_108,MLII,V1,1824,1763,1762,1,0,61,0,0
7,109,person_109,MLII,V1,2535,2532,2531,1,0,3,0,0
8,111,person_111,MLII,V1,2133,2124,2124,0,0,9,0,0
9,112,person_112,MLII,V1,2550,2539,2539,0,0,11,0,0



CELL 4 COMPLETE.


In [5]:
# ============================================================
# CELL 5: FULL CORRECTED DATASET COMPARISON
# ============================================================

# ------------------------------------------------------------
# 1. Prepare valid processed AAMI beats
# ------------------------------------------------------------

processed_valid = corrected_df.copy()

processed_valid["AAMI_Label"] = (
    processed_valid["Label"]
    .astype(str)
    .str.strip()
    .map(AAMI_MAP)
)

processed_valid = processed_valid[
    processed_valid["AAMI_Label"].notna()
].copy()

processed_valid["AAMI_Label"] = (
    processed_valid["AAMI_Label"].astype(str)
)

processed_valid = processed_valid.reset_index(drop=True)


print("Processed valid AAMI beats:", len(processed_valid))
print("Reconstructed beats:", len(reconstructed_df))


# ------------------------------------------------------------
# 2. Global class count comparison
# ------------------------------------------------------------

processed_global_counts = (
    processed_valid["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

rebuilt_global_counts = (
    reconstructed_df["AAMI_Label"]
    .value_counts()
    .reindex(AAMI_CLASSES, fill_value=0)
)

global_class_comparison = pd.DataFrame({

    "Expected": pd.Series(EXPECTED_CLASS_COUNTS),

    "Corrected_Processed": processed_global_counts,

    "Rebuilt_From_Original": rebuilt_global_counts

})

global_class_comparison["Processed_Matches_Expected"] = (
    global_class_comparison["Corrected_Processed"]
    == global_class_comparison["Expected"]
)

global_class_comparison["Rebuilt_Matches_Expected"] = (
    global_class_comparison["Rebuilt_From_Original"]
    == global_class_comparison["Expected"]
)

global_class_comparison["Processed_Matches_Rebuilt"] = (
    global_class_comparison["Corrected_Processed"]
    == global_class_comparison["Rebuilt_From_Original"]
)

print("\nGLOBAL AAMI CLASS COUNT COMPARISON")

display(global_class_comparison)


# ------------------------------------------------------------
# 3. Fingerprint helper
# ------------------------------------------------------------

def create_fingerprint_counters(df):

    waveform_values = df[
        WAVEFORM_COLUMNS
    ].to_numpy(dtype=np.float64)

    labels = df[
        "AAMI_Label"
    ].astype(str).to_numpy()

    waveform_counter = Counter()
    labelled_counter = Counter()

    digest_sequence = []

    for label, values in zip(labels, waveform_values):

        digest = waveform_digest(values)

        waveform_counter[digest] += 1

        labelled_counter[
            (label, digest)
        ] += 1

        digest_sequence.append(
            (label, digest)
        )

    return (
        waveform_counter,
        labelled_counter,
        digest_sequence
    )


# ------------------------------------------------------------
# 4. Compare every record independently
# ------------------------------------------------------------

record_comparison_rows = []
record_class_rows = []

for record_id in EXPECTED_RECORD_IDS:

    processed_record = processed_valid[
        processed_valid["Record_ID"] == record_id
    ].copy().reset_index(drop=True)

    rebuilt_record = reconstructed_df[
        reconstructed_df["Record_ID"] == record_id
    ].copy().reset_index(drop=True)


    # --------------------------------------------------------
    # A. Compare beat totals
    # --------------------------------------------------------

    processed_count = len(processed_record)
    rebuilt_count = len(rebuilt_record)

    beat_count_match = (
        processed_count == rebuilt_count
    )


    # --------------------------------------------------------
    # B. Compare five class counts within the record
    # --------------------------------------------------------

    class_match_flags = {}

    for label in AAMI_CLASSES:

        processed_class_count = int(
            (processed_record["AAMI_Label"] == label).sum()
        )

        rebuilt_class_count = int(
            (rebuilt_record["AAMI_Label"] == label).sum()
        )

        match = (
            processed_class_count == rebuilt_class_count
        )

        class_match_flags[label] = match

        record_class_rows.append({

            "Record_ID": record_id,

            "AAMI_Class": label,

            "Processed_Count": processed_class_count,

            "Rebuilt_Count": rebuilt_class_count,

            "Match": match

        })


    record_class_counts_match = all(
        class_match_flags.values()
    )


    # --------------------------------------------------------
    # C. Exact 402-value waveform comparison
    # --------------------------------------------------------

    (
        processed_wave_counter,
        processed_labelled_counter,
        processed_sequence
    ) = create_fingerprint_counters(processed_record)


    (
        rebuilt_wave_counter,
        rebuilt_labelled_counter,
        rebuilt_sequence
    ) = create_fingerprint_counters(rebuilt_record)


    exact_waveform_match = (
        processed_wave_counter == rebuilt_wave_counter
    )

    exact_waveform_and_label_match = (
        processed_labelled_counter == rebuilt_labelled_counter
    )


    # Count missing and extra waveform occurrences.
    missing_waveforms = sum(
        (processed_wave_counter - rebuilt_wave_counter).values()
    )

    extra_waveforms = sum(
        (rebuilt_wave_counter - processed_wave_counter).values()
    )

    missing_labelled_rows = sum(
        (processed_labelled_counter - rebuilt_labelled_counter).values()
    )

    extra_labelled_rows = sum(
        (rebuilt_labelled_counter - processed_labelled_counter).values()
    )


    # This is informational only. The main validation compares
    # exact waveform contents and multiplicities, not row order.
    ordered_sequence_match = (
        processed_sequence == rebuilt_sequence
    )


    # --------------------------------------------------------
    # D. Store record-level results
    # --------------------------------------------------------

    record_comparison_rows.append({

        "Record_ID": record_id,

        "Person_ID": get_person_id(record_id),

        "Processed_Valid_Beats": processed_count,

        "Rebuilt_Valid_Beats": rebuilt_count,

        "Beat_Count_Match": beat_count_match,

        "Record_Class_Counts_Match": record_class_counts_match,

        "Exact_402_Value_Waveforms": exact_waveform_match,

        "Exact_Waveforms_And_Labels": exact_waveform_and_label_match,

        "Missing_Waveforms": missing_waveforms,

        "Extra_Waveforms": extra_waveforms,

        "Missing_Labelled_Rows": missing_labelled_rows,

        "Extra_Labelled_Rows": extra_labelled_rows,

        "Ordered_Sequence_Match_Informational": ordered_sequence_match

    })


record_comparison_df = pd.DataFrame(
    record_comparison_rows
)

record_class_comparison_df = pd.DataFrame(
    record_class_rows
)


# ------------------------------------------------------------
# 5. Display record-level comparison
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("RECORD-LEVEL VERIFICATION")
print("=" * 75)

display(record_comparison_df)


# ------------------------------------------------------------
# 6. Display any mismatched recordings
# ------------------------------------------------------------

record_mismatches_df = record_comparison_df[
    (
        ~record_comparison_df["Beat_Count_Match"]
    )
    |
    (
        ~record_comparison_df["Record_Class_Counts_Match"]
    )
    |
    (
        ~record_comparison_df["Exact_402_Value_Waveforms"]
    )
    |
    (
        ~record_comparison_df["Exact_Waveforms_And_Labels"]
    )
].copy()

print("\nRECORDINGS WITH VALIDATION MISMATCHES:")

if record_mismatches_df.empty:

    print("None. All 46 recordings match.")

else:

    display(record_mismatches_df)


# ------------------------------------------------------------
# 7. Check record and person identity
# ------------------------------------------------------------

processed_record_ids = set(
    processed_valid["Record_ID"].unique()
)

rebuilt_record_ids = set(
    reconstructed_df["Record_ID"].unique()
)

record_id_sets_match = (
    processed_record_ids == rebuilt_record_ids
    == set(EXPECTED_RECORD_IDS)
)

processed_people = {
    get_person_id(r)
    for r in processed_record_ids
}

rebuilt_people = set(
    reconstructed_df["Person_ID"].unique()
)

person_count_match = (
    len(processed_people) == EXPECTED_PEOPLE
    and len(rebuilt_people) == EXPECTED_PEOPLE
    and processed_people == rebuilt_people
)

record201_202_shared = (
    get_person_id(201) == get_person_id(202)
)

print("\nRECORD AND PERSON CHECKS")
print("Processed recordings:", len(processed_record_ids))
print("Reconstructed recordings:", len(rebuilt_record_ids))
print("Person count:", len(rebuilt_people))
print("201 and 202 share Person_ID:", record201_202_shared)
print("Record ID sets match:", record_id_sets_match)
print("Person ID sets match:", person_count_match)


# ------------------------------------------------------------
# 8. Additional reconstruction integrity checks
# ------------------------------------------------------------

reconstructed_duplicate_keys = reconstructed_df.duplicated(
    ["Record_ID", "Sample_Index"]
).sum()

no_duplicate_sample_keys = (
    reconstructed_duplicate_keys == 0
)

no_parser_errors = (
    len(all_parser_errors) == 0
)

no_missing_internal_windows = (
    record_reconstruction_audit_df[
        "Missing_Internal_Windows"
    ].sum() == 0
)

no_incomplete_recordings = (
    len(record_reconstruction_audit_df) == EXPECTED_RECORDINGS
)

print("\nRECONSTRUCTION INTEGRITY")
print("Annotation parser errors:", len(all_parser_errors))
print("Duplicate record/sample keys:", reconstructed_duplicate_keys)

print(
    "Missing internal signal windows:",
    int(
        record_reconstruction_audit_df[
            "Missing_Internal_Windows"
        ].sum()
    )
)

print("All 46 records reconstructed:", no_incomplete_recordings)


# ------------------------------------------------------------
# 9. Final headline checks
# ------------------------------------------------------------

total_processed_valid = len(processed_valid)
total_rebuilt = len(reconstructed_df)

expected_total_pass = (
    total_processed_valid == EXPECTED_VALID_BEATS
    and total_rebuilt == EXPECTED_VALID_BEATS
)

expected_class_counts_pass = (
    processed_global_counts.to_dict()
    == EXPECTED_CLASS_COUNTS
    and rebuilt_global_counts.to_dict()
    == EXPECTED_CLASS_COUNTS
)

all_record_counts_pass = (
    record_comparison_df["Beat_Count_Match"].all()
)

all_record_class_counts_pass = (
    record_comparison_df[
        "Record_Class_Counts_Match"
    ].all()
)

all_402_waveforms_pass = (
    record_comparison_df[
        "Exact_402_Value_Waveforms"
    ].all()
)

all_waveforms_and_labels_pass = (
    record_comparison_df[
        "Exact_Waveforms_And_Labels"
    ].all()
)


# ------------------------------------------------------------
# 10. M1-B final validation gate
# ------------------------------------------------------------

M1B_CHECKS = {

    "Correction_Provenance": CORRECTION_PROVENANCE_PASSED,

    "Expected_Total_Beat_Count": expected_total_pass,

    "Expected_Global_Class_Counts": expected_class_counts_pass,

    "Record_ID_Sets": record_id_sets_match,

    "Person_ID_Sets": person_count_match,

    "201_202_Shared_Person_ID": record201_202_shared,

    "All_46_Records_Reconstructed": no_incomplete_recordings,

    "All_Record_Beat_Counts": all_record_counts_pass,

    "All_Record_Class_Counts": all_record_class_counts_pass,

    "Exact_402_Value_Waveforms": all_402_waveforms_pass,

    "Exact_Waveforms_And_AAMI_Labels": all_waveforms_and_labels_pass,

    "No_Annotation_Parser_Errors": no_parser_errors,

    "No_Duplicate_Record_Sample_Keys": no_duplicate_sample_keys,

    "No_Missing_Internal_Signal_Windows": no_missing_internal_windows

}


print("\n" + "=" * 75)
print("M1-B FULL VALIDATION GATE")
print("=" * 75)

for name, passed in M1B_CHECKS.items():

    print(
        f"{name}: {'PASS' if passed else 'FAIL'}"
    )


M1B_PASSED = all(
    M1B_CHECKS.values()
)


print("\n" + "=" * 75)

if M1B_PASSED:

    print("M1-B FINAL STATUS: PASS")

    print(
        "All 105,047 valid beats match the reconstructed "
        "original ECG data."
    )

else:

    print("M1-B FINAL STATUS: NOT PASSED")

    print(
        "Do not proceed to M0 baseline retraining or M2."
    )

print("=" * 75)

Processed valid AAMI beats: 105047
Reconstructed beats: 105047

GLOBAL AAMI CLASS COUNT COMPARISON


,Expected,Corrected_Processed,Rebuilt_From_Original,Processed_Matches_Expected,Rebuilt_Matches_Expected,Processed_Matches_Rebuilt
N,90341,90341,90341,True,True,True
S,2781,2781,2781,True,True,True
V,7229,7229,7229,True,True,True
F,802,802,802,True,True,True
Q,3894,3894,3894,True,True,True



RECORD-LEVEL VERIFICATION


,Record_ID,Person_ID,Processed_Valid_Beats,Rebuilt_Valid_Beats,Beat_Count_Match,Record_Class_Counts_Match,Exact_402_Value_Waveforms,Exact_Waveforms_And_Labels,Missing_Waveforms,Extra_Waveforms,Missing_Labelled_Rows,Extra_Labelled_Rows,Ordered_Sequence_Match_Informational
0,100,person_100,2271,2271,True,True,True,True,0,0,0,0,True
1,101,person_101,1864,1864,True,True,True,True,0,0,0,0,True
2,103,person_103,2084,2084,True,True,True,True,0,0,0,0,True
3,105,person_105,2572,2572,True,True,True,True,0,0,0,0,True
4,106,person_106,2027,2027,True,True,True,True,0,0,0,0,True
5,107,person_107,2136,2136,True,True,True,True,0,0,0,0,True
6,108,person_108,1762,1762,True,True,True,True,0,0,0,0,True
7,109,person_109,2531,2531,True,True,True,True,0,0,0,0,True
8,111,person_111,2124,2124,True,True,True,True,0,0,0,0,True
9,112,person_112,2539,2539,True,True,True,True,0,0,0,0,True



RECORDINGS WITH VALIDATION MISMATCHES:
None. All 46 recordings match.

RECORD AND PERSON CHECKS
Processed recordings: 46
Reconstructed recordings: 46
Person count: 45
201 and 202 share Person_ID: True
Record ID sets match: True
Person ID sets match: True

RECONSTRUCTION INTEGRITY
Annotation parser errors: 0
Duplicate record/sample keys: 0
Missing internal signal windows: 0
All 46 records reconstructed: True

M1-B FULL VALIDATION GATE
Correction_Provenance: PASS
Expected_Total_Beat_Count: PASS
Expected_Global_Class_Counts: PASS
Record_ID_Sets: PASS
Person_ID_Sets: PASS
201_202_Shared_Person_ID: PASS
All_46_Records_Reconstructed: PASS
All_Record_Beat_Counts: PASS
All_Record_Class_Counts: PASS
Exact_402_Value_Waveforms: PASS
Exact_Waveforms_And_AAMI_Labels: PASS
No_Annotation_Parser_Errors: PASS
No_Duplicate_Record_Sample_Keys: PASS
No_Missing_Internal_Signal_Windows: PASS

M1-B FINAL STATUS: PASS
All 105,047 valid beats match the reconstructed original ECG data.


In [6]:
# ============================================================
# CELL 6: SAVE M1-B VERIFICATION OUTPUTS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Create output folder
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/ECG_Project/04_M1B_Verification"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output folder:", OUTPUT_DIR)


# ------------------------------------------------------------
# 2. Combine reconstruction and comparison audits
# ------------------------------------------------------------

complete_record_audit = (
    record_reconstruction_audit_df.merge(
        record_comparison_df,
        on=["Record_ID", "Person_ID"],
        how="left",
        validate="one_to_one"
    )
)

complete_record_audit.to_csv(
    OUTPUT_DIR / "m1b_record_verification_audit.csv",
    index=False
)

global_class_comparison.to_csv(
    OUTPUT_DIR / "m1b_global_class_count_comparison.csv"
)

record_class_comparison_df.to_csv(
    OUTPUT_DIR / "m1b_record_class_count_comparison.csv",
    index=False
)

record_mismatches_df.to_csv(
    OUTPUT_DIR / "m1b_record_mismatches.csv",
    index=False
)

source_header_audit_df.to_csv(
    OUTPUT_DIR / "m1b_original_source_file_audit.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Save parser errors, if any
# ------------------------------------------------------------

parser_errors_df = pd.DataFrame(
    all_parser_errors
)

parser_errors_df.to_csv(
    OUTPUT_DIR / "m1b_annotation_parser_errors.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Save reconstructed beat table
# ------------------------------------------------------------

if M1B_PASSED:

    reconstructed_output_path = (
        OUTPUT_DIR /
        "validated_reconstructed_beat_table.csv.gz"
    )

else:

    reconstructed_output_path = (
        OUTPUT_DIR /
        "diagnostic_reconstructed_beat_table.csv.gz"
    )


reconstructed_df.to_csv(
    reconstructed_output_path,
    index=False,
    compression="gzip",
    float_format="%.17g"
)


# ------------------------------------------------------------
# 5. Save verification summary
# ------------------------------------------------------------

verification_summary = {

    "Project": "ECG Beat Review and Annotation Assistant",

    "Phase": "M1-B",

    "Account_Result": (
        "PASS" if M1B_PASSED else "NOT PASSED"
    ),

    "Original_Dataset_Path": original_dataset_path,

    "Corrected_Dataset_Path": corrected_dataset_path,

    "Expected_Valid_Beats": EXPECTED_VALID_BEATS,

    "Processed_Valid_Beats": int(total_processed_valid),

    "Reconstructed_Valid_Beats": int(total_rebuilt),

    "Expected_Recordings": EXPECTED_RECORDINGS,

    "Reconstructed_Recordings": int(
        reconstructed_df["Record_ID"].nunique()
    ),

    "Expected_People": EXPECTED_PEOPLE,

    "Reconstructed_People": int(
        reconstructed_df["Person_ID"].nunique()
    ),

    "Expected_AAMI_Class_Counts": EXPECTED_CLASS_COUNTS,

    "Processed_AAMI_Class_Counts": {
        k: int(v)
        for k, v in processed_global_counts.to_dict().items()
    },

    "Reconstructed_AAMI_Class_Counts": {
        k: int(v)
        for k, v in rebuilt_global_counts.to_dict().items()
    },

    "M1B_Checks": {
        k: bool(v)
        for k, v in M1B_CHECKS.items()
    },

    "M1B_Passed": bool(M1B_PASSED),

    "Record_114_Correction_Verified": bool(
        record114_mlii_correct and record114_lead2_correct
    ),

    "Total_Annotation_Parser_Errors": int(
        len(all_parser_errors)
    ),

    "Total_Duplicate_Record_Sample_Keys": int(
        reconstructed_duplicate_keys
    ),

    "Total_Missing_Internal_Windows": int(
        record_reconstruction_audit_df[
            "Missing_Internal_Windows"
        ].sum()
    ),

    "Reconstructed_Beat_Table": str(
        reconstructed_output_path
    )

}


with open(
    OUTPUT_DIR / "m1b_verification_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        verification_summary,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 6. Save file hashes for reproducibility
# ------------------------------------------------------------

file_manifest = {

    "Original_Processed_Dataset_SHA256": sha256_file(
        original_dataset_path
    ),

    "Corrected_Processed_Dataset_SHA256": sha256_file(
        corrected_dataset_path
    ),

    "Original_Dataset_Path": original_dataset_path,

    "Corrected_Dataset_Path": corrected_dataset_path

}


with open(
    OUTPUT_DIR / "m1b_file_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        file_manifest,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 7. Final output
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("M1-B VERIFICATION OUTPUTS SAVED")
print("=" * 75)

for output_file in sorted(OUTPUT_DIR.iterdir()):

    if output_file.is_file():

        size_mb = output_file.stat().st_size / (1024 ** 2)

        print(
            f"{output_file.name} — {size_mb:.2f} MB"
        )


print("\n" + "=" * 75)

if M1B_PASSED:

    print("M1-B FINAL RESULT: PASS")

    print("\nAll verification checks passed.")

    print(
        "The reconstructed beat table is saved as "
        "validated_reconstructed_beat_table.csv.gz"
    )

    print("\nM1-B is ready for project-gate review.")

else:

    print("M1-B FINAL RESULT: NOT PASSED")

    print(
        "\nThe reconstructed table is saved for diagnosis, "
        "not as a validated dataset."
    )

    print(
        "Review m1b_record_mismatches.csv and "
        "m1b_verification_summary.json."
    )

print("\nNo input datasets were modified.")
print("No models were trained.")
print("No M0 splits were changed.")

print("=" * 75)

Output folder: /kaggle/working/ECG_Project/04_M1B_Verification

M1-B VERIFICATION OUTPUTS SAVED
m1b_annotation_parser_errors.csv — 0.00 MB
m1b_file_manifest.json — 0.00 MB
m1b_global_class_count_comparison.csv — 0.00 MB
m1b_original_source_file_audit.csv — 0.01 MB
m1b_record_class_count_comparison.csv — 0.00 MB
m1b_record_mismatches.csv — 0.00 MB
m1b_record_verification_audit.csv — 0.00 MB
m1b_verification_summary.json — 0.00 MB
validated_reconstructed_beat_table.csv.gz — 41.84 MB

M1-B FINAL RESULT: PASS

All verification checks passed.
The reconstructed beat table is saved as validated_reconstructed_beat_table.csv.gz

M1-B is ready for project-gate review.

No input datasets were modified.
No models were trained.
No M0 splits were changed.
